# Model Deployment

This notebook demonstrates how to deploy the trained AI model for various applications. It covers:

1. Loading the trained model
2. Optimizing the model for deployment
3. Exporting the model for Flask applications
4. Converting the model to Core ML format for Apple devices
5. Testing the deployed model

## Setup

First, let's install the required packages and import the necessary modules.

In [ ]:
# Install required packages
!pip install transformers torch onnx optimum pyyaml flask flask-cors

In [ ]:
# Import modules
import os
import sys
import logging
import yaml
import json
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# Add src directory to path
sys.path.append(os.path.abspath('../'))

# Import custom modules
from src.deployment.flask_export import FlaskModelExporter
from src.deployment.coreml_export import CoreMLModelExporter

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Load Configuration

Load the configuration file that defines the deployment parameters.

In [ ]:
# Load configuration
with open("../configs/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# Display deployment configuration
print("Flask deployment configuration:")
for key, value in config['deployment']['flask'].items():
    print(f"  {key}: {value}")

print("\nCore ML deployment configuration:")
for key, value in config['deployment']['coreml'].items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for subkey, subvalue in value.items():
            print(f"    {subkey}: {subvalue}")
    else:
        print(f"  {key}: {value}")

## Load Model

Load the trained model for deployment.

In [ ]:
# Define model path
model_path = "../outputs/final_model"

# Check if model exists
if not os.path.exists(model_path):
    print(f"Model not found at {model_path}. Please run the model training notebook first.")
    # For demonstration, load a pretrained model
    model_path = "gpt2"
    print(f"Loading pretrained model {model_path} for demonstration.")

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print(f"Loaded model from {model_path}")
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## Optimize Model for Deployment

Optimize the model for deployment by applying quantization and pruning.

In [ ]:
def quantize_model(model, quantization_config):
    """Quantize the model to reduce size."""
    if not quantization_config['enabled']:
        return model
    
    precision = quantization_config['precision']
    print(f"Quantizing model to {precision} precision")
    
    try:
        from optimum.onnxruntime import ORTQuantizer
        from optimum.onnxruntime.configuration import AutoQuantizationConfig
        
        # Export model to ONNX format first
        onnx_path = "temp_model.onnx"
        
        # Create dummy input
        batch_size = 1
        seq_len = 16
        input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, seq_len))
        attention_mask = torch.ones_like(input_ids)
        
        # Export to ONNX
        torch.onnx.export(
            model,
            (input_ids, attention_mask),
            onnx_path,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            dynamic_axes={
                'input_ids': {0: 'batch_size', 1: 'sequence_length'},
                'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
                'logits': {0: 'batch_size', 1: 'sequence_length'}
            },
            opset_version=12
        )
        
        # Quantize the model
        quantizer = ORTQuantizer.from_pretrained(model_path, file_name=onnx_path)
        qconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=False)
        quantized_model = quantizer.quantize(qconfig)
        
        print("Model quantized successfully")
        return quantized_model
    except Exception as e:
        print(f"Error quantizing model: {e}")
        print("Returning original model")
        return model

def prune_model(model, pruning_config):
    """Prune the model to reduce size."""
    if not pruning_config['enabled']:
        return model
    
    method = pruning_config['method']
    target_sparsity = pruning_config['target_sparsity']
    
    print(f"Pruning model using {method} method with target sparsity {target_sparsity}")
    
    try:
        import torch.nn.utils.prune as prune
        
        # Apply pruning to linear layers
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear):
                if method == 'magnitude':
                    prune.l1_unstructured(module, name='weight', amount=target_sparsity)
                elif method == 'random':
                    prune.random_unstructured(module, name='weight', amount=target_sparsity)
        
        # Make pruning permanent
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear):
                prune.remove(module, 'weight')
        
        print("Model pruned successfully")
        return model
    except Exception as e:
        print(f"Error pruning model: {e}")
        print("Returning original model")
        return model

# Optimize model for deployment
print("Optimizing model for deployment...")

# Apply quantization
quantized_model = quantize_model(model, config['deployment']['optimization']['quantization'])

# Apply pruning
optimized_model = prune_model(quantized_model, config['deployment']['optimization']['pruning'])

print(f"Original model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print(f"Optimized model size: {sum(p.numel() for p in optimized_model.parameters()) / 1e6:.2f}M parameters")

## Export Model for Flask Applications

Export the optimized model for use in Flask applications.

In [ ]:
# Create output directory
output_dir = "../outputs/deployment"
os.makedirs(output_dir, exist_ok=True)

# Export model for Flask
print("Exporting model for Flask applications...")

flask_exporter = FlaskModelExporter(
    model=optimized_model,
    tokenizer=tokenizer,
    output_dir=output_dir,
    model_name="ai_model_flask",
    config=config['deployment']
)

flask_export_dir = flask_exporter.export()

print(f"Model exported for Flask applications at {flask_export_dir}")

## Convert Model to Core ML Format

Convert the optimized model to Core ML format for Apple devices.

In [ ]:
# Export model to Core ML format
print("Exporting model to Core ML format...")

coreml_exporter = CoreMLModelExporter(
    model=optimized_model,
    tokenizer=tokenizer,
    output_dir=output_dir,
    model_name="ai_model_coreml",
    config=config['deployment']
)

coreml_export_dir = coreml_exporter.export()

print(f"Model exported to Core ML format at {coreml_export_dir}")

## Test Flask Deployment

Test the Flask deployment by running the Flask application and making requests.

In [ ]:
# Test Flask deployment
print("Testing Flask deployment...")

# Start Flask server in background
import subprocess
import time
import requests

# Change to Flask export directory
os.chdir(flask_export_dir)

# Start Flask server
try:
    # Start server in background
    server_process = subprocess.Popen(["python", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    
    # Wait for server to start
    print("Waiting for Flask server to start...")
    time.sleep(10)
    
    # Test health endpoint
    health_response = requests.get("http://localhost:5000/health")
    print(f"Health endpoint response: {health_response.json()}")
    
    # Test info endpoint
    info_response = requests.get("http://localhost:5000/info")
    print(f"Info endpoint response: {info_response.json()}")
    
    # Test generate endpoint
    generate_data = {
        "prompt": "Write a function to calculate the factorial of a number:",
        "max_length": 100,
        "temperature": 0.7
    }
    
    generate_response = requests.post("http://localhost:5000/generate", json=generate_data)
    generate_result = generate_response.json()
    
    print(f"Generate endpoint response:")
    print(f"  Prompt: {generate_result['prompt']}")
    print(f"  Generated text: {generate_result['generated_text']}")
    print(f"  Generation time: {generate_result['generation_time']:.2f} seconds")
    
except Exception as e:
    print(f"Error testing Flask deployment: {e}")
finally:
    # Stop Flask server
    server_process.terminate()
    print("Flask server stopped")
    
    # Change back to notebook directory
    os.chdir("../notebooks")

## Create Deployment Documentation

Create documentation for deploying the model in different environments.

In [ ]:
# Create deployment documentation
deployment_doc = f"""
# AI Model Deployment Guide

This guide provides instructions for deploying the trained AI model in different environments.

## Flask Deployment

The model has been exported for use in Flask applications at `{flask_export_dir}`.

### Setup

1. Install the required packages:

```bash
pip install -r {flask_export_dir}/requirements.txt
```

2. Run the Flask application:

```bash
cd {flask_export_dir}
python app.py
```

The application will be available at http://localhost:5000.

### API Endpoints

- `/generate`: Generate text from a prompt
- `/generate_async`: Generate text asynchronously
- `/health`: Check if the model is loaded and ready
- `/info`: Get information about the model

### Production Deployment

For production deployment, it is recommended to use a WSGI server such as Gunicorn:

```bash
pip install gunicorn
cd {flask_export_dir}
gunicorn -w 4 -b 0.0.0.0:5000 app:app
```

## Core ML Deployment

The model has been exported to Core ML format for Apple devices at `{coreml_export_dir}`.

### Integration with iOS/macOS Applications

1. Add the `.mlmodel` file to your Xcode project
2. Use the example code in `{coreml_export_dir}/ModelExample.swift` as a starting point

### Requirements

- iOS 14.0+ / macOS 11.0+
- Xcode 12.0+
- Swift 5.3+

### Performance Considerations

- The model performs best on devices with Neural Engine
- For older devices, consider using a smaller model or reducing the generation length
- Use batching for multiple generations to improve throughput

## Model Optimization

The deployed model has been optimized with the following techniques:

- Quantization: {config['deployment']['optimization']['quantization']['enabled']}
- Pruning: {config['deployment']['optimization']['pruning']['enabled']}
- Distillation: {config['deployment']['optimization']['distillation']['enabled']}

These optimizations reduce the model size and improve inference speed while maintaining acceptable performance.
"""

# Save deployment documentation
with open(os.path.join(output_dir, "deployment_guide.md"), 'w') as f:
    f.write(deployment_doc)

print(f"Deployment documentation saved to {os.path.join(output_dir, 'deployment_guide.md')}")

## Summary

In this notebook, we have:

1. Loaded the trained model
2. Optimized the model for deployment using quantization and pruning
3. Exported the model for Flask applications
4. Converted the model to Core ML format for Apple devices
5. Tested the Flask deployment
6. Created deployment documentation

The model is now ready for deployment in various environments, including web applications via Flask and mobile devices via Core ML.